# 1. Решение

In [6]:
import nglview as nv # Опционально, если хочется посмотреть на .pdb
from rdkit import Chem
from pathlib import Path

from openff.toolkit import ForceField, Molecule, Topology

In [ ]:
def openff_setup(mol, name):

    '''Функция openff_setup() параметризует систему и сохраняет файлы .pdb и .top.
    Cиловое поле фиксировано.'''

    openff_mol = Molecule.from_rdkit(mol)
    
    mol_top = Topology.from_molecules(openff_mol)
    sage = ForceField("openff-2.2.0.offxml")
    
    interchange = sage.create_interchange(mol_top)
    
    # ! mkdir -p results/$name # Работает, если код запускается в Jupyter Notebook :)
    

    o_dir = Path(f"results/{name}") # Версия без использования magic commands - надо импортировать Path
    o_dir.mkdir(parents=True, exist_ok=True)

    interchange.to_top(f"results/{name}/{name}.top")
    interchange.to_pdb(f"results/{name}/{name}.pdb")


def read_and_create(path):

    '''Функция read_and_create() читает .sdf файл, перенаправляет объект RDKit с молекулой 
    в функцию openff_setup() для параметризации'''
    
    # !mkdir -p results
    
    if not Path("results").exists(): # Папка с аутпутом
        Path("results").mkdir()

    
    suppl = Chem.SDMolSupplier(path)
    for i,mol in enumerate(suppl,start=1):
        if mol is None:
            print(f"Warning: molecule number {i} is None, check input file")
            continue

        # Обработка исключения: названия молекулы нет, если не соблюдается структура заголовка .sdf файла
        try:
            mol_name = mol.GetProp("_Name")
            if mol_name.strip() == '':
                mol_name = f'Unnamed_{i}'
        except:
            raise ValueError('Wrong structure of SDF file: no Header') 
            
        openff_setup(mol, mol_name)
    

read_and_create('openff_data/example.sdf')

# 2. Проверка output

- Отображение `.pdb` файла в nglview
- Проверка `.top` (создается ли файл .tpr)

In [2]:
view = nv.show_structure_file('results/CHEMBL45305/CHEMBL45305.pdb')
view

NGLWidget()

In [3]:
%%bash
cp -r results/CHEMBL45305/* tmp/
echo '''; Минимальный mdp-файл для проверки топологии
integrator = steep
nsteps     = 0 ; 0 шагов минимизации''' > tmp/test.tpr

In [5]:
%%bash
cd tmp
gmx editconf -f CHEMBL45305.pdb -o boxed.gro -c -d 2.0 -bt cubic # Задаем параметры коробки
gmx grompp -f test.mdp -c boxed.gro -p CHEMBL45305.top -o test.tpr -maxwarn 10

               :-) GROMACS - gmx editconf, 2026.0-conda_forge (-:

Executable:   /home/nuta/miniconda3/envs/gmx/bin.AVX2_256/gmx
Data prefix:  /home/nuta/miniconda3/envs/gmx
Working dir:  /home/nuta/1_task/tmp
Command line:
  gmx editconf -f CHEMBL45305.pdb -o boxed.gro -c -d 2.0 -bt cubic


GROMACS reminds you: "Computers are like humans - they do everything except think." (John von Neumann)



Note that major changes are planned in future for editconf, to improve usability and utility.
Read 49 atoms
No velocities found
    system size :  0.802  1.448  0.706 (nm)
    diameter    :  1.576               (nm)
    center      :  0.004  0.003  0.005 (nm)
    box vectors :  0.000  0.000  0.000 (nm)
    box angles  :   0.00   0.00   0.00 (degrees)
    box volume  :   0.00               (nm^3)
    shift       :  2.784  2.785  2.783 (nm)
new center      :  2.788  2.788  2.788 (nm)
new box vectors :  5.576  5.576  5.576 (nm)
new box angles  :  90.00  90.00  90.00 (degrees)
new box volume  : 173.38               (nm^3)


                :-) GROMACS - gmx grompp, 2026.0-conda_forge (-:

Executable:   /home/nuta/miniconda3/envs/gmx/bin.AVX2_256/gmx
Data prefix:  /home/nuta/miniconda3/envs/gmx
Working dir:  /home/nuta/1_task/tmp
Command line:
  gmx grompp -f test.mdp -c boxed.gro -p CHEMBL45305.top -o test.tpr -maxwarn 10


NOTE 1 [file test.mdp]:
  For a correct single-point energy evaluation with nsteps = 0, use
  continuation = yes to avoid constraining the input coordinates.

Generating 1-4 interactions: fudge = 0.5
atom name 1 in CHEMBL45305.top and boxed.gro does not match (Cl - Cl1x)

atom name 2 in CHEMBL45305.top and boxed.gro does not match (C - C1x)

atom name 3 in CHEMBL45305.top and boxed.gro does not match (C - C2x)

atom name 4 in CHEMBL45305.top and boxed.gro does not match (C - C3x)

atom name 5 in CHEMBL45305.top and boxed.gro does not match (F - F1x)

atom name 6 in CHEMBL45305.top and boxed.gro does not match (C - C4x)

atom name 7 in CHEMBL45305.top and boxed.gro does not match (C - C

Setting the LD random seed to -1251509457

Generated 1225 of the 1225 non-bonded parameter combinations

Generated 1225 of the 1225 1-4 parameter combinations

Excluding 3 bonded neighbours molecule type 'CHEMBL45305'

atom name 1 in CHEMBL45305.top and boxed.gro does not match (Cl - Cl1x)

atom name 2 in CHEMBL45305.top and boxed.gro does not match (C - C1x)

atom name 3 in CHEMBL45305.top and boxed.gro does not match (C - C2x)

atom name 4 in CHEMBL45305.top and boxed.gro does not match (C - C3x)

atom name 5 in CHEMBL45305.top and boxed.gro does not match (F - F1x)

atom name 6 in CHEMBL45305.top and boxed.gro does not match (C - C4x)

atom name 7 in CHEMBL45305.top and boxed.gro does not match (C - C5x)

atom name 8 in CHEMBL45305.top and boxed.gro does not match (C - C6x)

atom name 9 in CHEMBL45305.top and boxed.gro does not match (C - C7x)

atom name 10 in CHEMBL45305.top and boxed.gro does not match (O - O1x)

atom name 11 in CHEMBL45305.top and boxed.gro does not match (N - N1